# HW02 Part 1: Download Hourly Wikipedia Data

This notebook downloads the hourly Wikipedia event data in Parquet format from the course S3 bucket into the local `data` directory.

In [1]:
from pathlib import Path

import boto3

In [2]:
# define the S3 bucket and prefix for the source data

source_bucket = "dsan6000-wikipedia"
source_prefix = "hourly_parquet/"
data_dir = Path("data")

data_dir.mkdir(exist_ok=True)

s3 = boto3.client("s3")

In [3]:
# download all Parquet files from the S3 bucket to the local data directory

downloaded_files = []

paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(
    Bucket=source_bucket,
    Prefix=source_prefix,
):
    for obj in page.get("Contents", []):
        s3_key = obj["Key"]

        if s3_key.endswith(".parquet"):
            local_path = data_dir / Path(s3_key).name
            s3.download_file(source_bucket, s3_key, str(local_path))
            downloaded_files.append(local_path.name)

print(f"Downloaded {len(downloaded_files)} Parquet files.")

Downloaded 24 Parquet files.


In [4]:
# verification 

parquet_files = sorted(data_dir.glob("*.parquet"))
total_size_mb = sum(file.stat().st_size for file in parquet_files) / (1024**2)

print(f"Local Parquet file count: {len(parquet_files)}")
print(f"Total size: {total_size_mb:.2f} MB")
print("First five files:")

for file in parquet_files[:5]:
    print(file.name)

Local Parquet file count: 24
Total size: 30.53 MB
First five files:
20260901_040000.parquet
20260901_050000.parquet
20260901_060000.parquet
20260901_070000.parquet
20260901_080000.parquet
